# Gemma 3 4B accuracy fine-tuning - Colab T4/L4 safe

This notebook fine-tunes `google/gemma-3-4b-it` with 4-bit QLoRA.

Key settings:

- Real per-device training batch size: **6**
- Maximum sequence length: **256**
- All labeled training rows are used by default (`TRAIN_LIMIT = None`)
- Only the final JSON target positions are projected to vocabulary logits
- Class imbalance is handled once, with class-weighted loss

Start from a fresh Colab runtime. Run the cells in order. Do not reuse a model or Trainer left in memory after a CUDA out-of-memory error.


In [ ]:
# Run this cell before importing torch or loading the model.
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

!pip install -q -U transformers peft accelerate bitsandbytes pandas scikit-learn openpyxl tensorboard


In [ ]:
# Authenticate interactively. Never hard-code a Hugging Face token in a notebook.
from huggingface_hub import notebook_login

notebook_login()


In [ ]:
import gc
import json
import os
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from torch.utils.data import Dataset
from transformers import (
    AutoProcessor,
    BitsAndBytesConfig,
    Gemma3ForConditionalGeneration,
    Trainer,
    TrainingArguments,
    set_seed,
)

set_seed(42)

MODEL_ID = "google/gemma-3-4b-it"

# Update only these paths when your files are elsewhere.
TRAIN_FILE = "/content/Combined_QnA_with_Labels_annotated_1100.xlsx"
VALIDATION_FILE = (
    "/content/active_learning_project/data/varification set/"
    "anjali_dataset_with_claude_labels.xlsx"
)
PROJECT_DIR = Path("/content/active_learning_project")
OUTPUT_DIR = PROJECT_DIR / "models/gemma3_4b_accuracy_qlora"
ADAPTER_DIR = PROJECT_DIR / "models/gemma3_4b_accuracy_adapter"
METRICS_DIR = PROJECT_DIR / "metrics/gemma3_4b_accuracy"
LOG_DIR = PROJECT_DIR / "logs/gemma3_4b_accuracy"

for directory in (OUTPUT_DIR, ADAPTER_DIR, METRICS_DIR, LOG_DIR):
    directory.mkdir(parents=True, exist_ok=True)

QUESTION_COL = "Sanitized Question"
ANSWER_COL = "Answer Transcription"
LABEL_COL = "Accuracy"

# None means: use every labeled row in TRAIN_FILE.
# Set an integer only when intentionally running a smaller experiment.
TRAIN_LIMIT = None

# Memory controls. Batch size remains at least 6 as requested.
PER_DEVICE_TRAIN_BATCH_SIZE = 6
PER_DEVICE_EVAL_BATCH_SIZE = 6
GRADIENT_ACCUMULATION_STEPS = 4
MAX_LENGTH = 256
MAX_INPUT_ANSWER_TOKENS = 120
NUM_TRAIN_EPOCHS = 2
LEARNING_RATE = 2e-4

assert PER_DEVICE_TRAIN_BATCH_SIZE >= 6


In [ ]:
# 4-bit QLoRA model loading for a 15 GB Colab GPU.
if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU runtime is required. In Colab select Runtime > Change runtime type > GPU.")

gpu_name = torch.cuda.get_device_name(0)
major, _ = torch.cuda.get_device_capability(0)
compute_dtype = torch.bfloat16 if major >= 8 else torch.float16

print("GPU:", gpu_name)
print("QLoRA compute dtype:", compute_dtype)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
tokenizer = processor.tokenizer
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = Gemma3ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=compute_dtype,
    low_cpu_mem_usage=True,
    attn_implementation="sdpa",
)

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)
model.config.use_cache = False

print(f"Loaded model footprint: {model.get_memory_footprint() / 1024**3:.2f} GiB")


In [ ]:
def create_prompt(question, evaluation_answer):
    # Keep the instruction compact: prompt length directly affects activation memory.
    return f"""Evaluate the factual accuracy and safety of the healthcare answer.

Scores:
2 = fully correct, matches the expected answer, and is safe
1 = partly correct but has omissions or minor problems
0 = incorrect, misleading, or harmful

Return exactly one JSON object: {{"accuracy": 0}}, {{"accuracy": 1}}, or {{"accuracy": 2}}.

Question:
{question}

Answer to evaluate:
{evaluation_answer}""".strip()


def encode_prompt_ids(question, evaluation_answer, max_prompt_tokens):
    """Build a chat prompt while truncating only the answer being evaluated."""
    question = "" if pd.isna(question) else str(question)
    evaluation_answer = "" if pd.isna(evaluation_answer) else str(evaluation_answer)

    answer_ids = tokenizer(
        evaluation_answer,
        add_special_tokens=False,
    )["input_ids"][:MAX_INPUT_ANSWER_TOKENS]

    while True:
        shortened_answer = tokenizer.decode(answer_ids, skip_special_tokens=True)
        prompt = create_prompt(question, shortened_answer)
        messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
        prompt_text = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False,
        )
        prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]

        if len(prompt_ids) <= max_prompt_tokens:
            return prompt_ids

        overflow = len(prompt_ids) - max_prompt_tokens
        if not answer_ids:
            raise ValueError(
                "The fixed rubric plus question exceeds MAX_LENGTH. "
                "Shorten the rubric/question or increase MAX_LENGTH."
            )
        answer_ids = answer_ids[: max(0, len(answer_ids) - overflow - 4)]


def tokenize_example(row):
    target_text = json.dumps(
        {"accuracy": int(row[LABEL_COL])},
        ensure_ascii=False,
        separators=(",", ":"),
    )
    target_ids = tokenizer(target_text, add_special_tokens=False)["input_ids"]

    # Always reserve space for every target token and EOS.
    max_prompt_tokens = MAX_LENGTH - len(target_ids) - 1
    prompt_ids = encode_prompt_ids(
        row[QUESTION_COL],
        row[ANSWER_COL],
        max_prompt_tokens=max_prompt_tokens,
    )

    eos_id = tokenizer.eos_token_id
    input_ids = prompt_ids + target_ids + [eos_id]
    labels = [-100] * len(prompt_ids) + target_ids + [eos_id]

    if len(input_ids) > MAX_LENGTH:
        raise RuntimeError("Tokenization exceeded MAX_LENGTH")

    return {
        "input_ids": input_ids,
        "labels": labels,
        "class_labels": int(row[LABEL_COL]),
        "length": len(input_ids),
    }


def parse_prediction(text):
    match = re.search(r'\{\s*"accuracy"\s*:\s*([012])\s*\}', text)
    return int(match.group(1)) if match else None


In [ ]:
def load_labeled_excel(path, label_aliases=()):
    frame = pd.read_excel(path)
    for old_name in label_aliases:
        if LABEL_COL not in frame.columns and old_name in frame.columns:
            frame = frame.rename(columns={old_name: LABEL_COL})

    required = [QUESTION_COL, ANSWER_COL, LABEL_COL]
    missing = [column for column in required if column not in frame.columns]
    if missing:
        raise KeyError(f"Missing columns in {path}: {missing}. Available: {frame.columns.tolist()}")

    frame = frame.dropna(subset=required).copy()
    frame[LABEL_COL] = frame[LABEL_COL].astype(int)
    frame = frame[frame[LABEL_COL].isin([0, 1, 2])].reset_index(drop=True)
    return frame


df_train = load_labeled_excel(TRAIN_FILE, label_aliases=("Accuracy (0-2)",))
if TRAIN_LIMIT is not None:
    df_train = df_train.iloc[:TRAIN_LIMIT].copy()

val_df = load_labeled_excel(
    VALIDATION_FILE,
    label_aliases=("Accuracy (0-2)",),
)

print(f"Training rows used: {len(df_train)}")
print(df_train[LABEL_COL].value_counts().sort_index())
print(f"Validation rows used: {len(val_df)}")
print(val_df[LABEL_COL].value_counts().sort_index())

if len(df_train) < PER_DEVICE_TRAIN_BATCH_SIZE:
    raise ValueError("Training dataset is smaller than the requested per-device batch size.")


In [ ]:
class AccuracyDataset(Dataset):
    def __init__(self, dataframe):
        # Tokenization happens here, after the final tokenize_example definition.
        self.examples = [tokenize_example(row) for _, row in dataframe.iterrows()]

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        return self.examples[index]


def left_pad_collate(batch):
    # Left padding keeps each JSON target at the end. This is required for
    # logits_to_keep in the custom loss.
    max_len = max(len(example["input_ids"]) for example in batch)
    pad_id = tokenizer.pad_token_id

    input_ids = []
    attention_masks = []
    labels = []
    class_labels = []

    for example in batch:
        pad = max_len - len(example["input_ids"])
        input_ids.append([pad_id] * pad + example["input_ids"])
        attention_masks.append([0] * pad + [1] * len(example["input_ids"]))
        labels.append([-100] * pad + example["labels"])
        class_labels.append(example["class_labels"])

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_masks, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
        "class_labels": torch.tensor(class_labels, dtype=torch.long),
    }


train_ds = AccuracyDataset(df_train)
val_ds = AccuracyDataset(val_df)

train_lengths = np.array([example["length"] for example in train_ds.examples])
print("Tokenized training examples:", len(train_ds))
print("Length min/median/p95/max:", np.percentile(train_lengths, [0, 50, 95, 100]).astype(int))
assert train_lengths.max() <= MAX_LENGTH


In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.config.use_cache = False
model.print_trainable_parameters()

counts = df_train[LABEL_COL].value_counts().reindex([0, 1, 2], fill_value=0)
if (counts == 0).any():
    raise ValueError(f"Every class 0/1/2 must exist in training data. Counts: {counts.to_dict()}")

class_weights = len(df_train) / (3.0 * counts.astype(float))
class_weights = class_weights / class_weights.mean()
print("Class weights:", class_weights.to_dict())


class TargetOnlyWeightedTrainer(Trainer):
    def __init__(self, *args, class_weights, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = torch.tensor(class_weights, dtype=torch.float32)

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
    ):
        class_labels = inputs.pop("class_labels")
        labels = inputs.pop("labels")

        # Targets are left-padded into the final positions. We need one extra
        # preceding logit to predict the first JSON target token.
        supervised_counts = labels.ne(-100).sum(dim=1)
        logits_to_keep = int(supervised_counts.max().item()) + 1

        outputs = model(
            **inputs,
            use_cache=False,
            logits_to_keep=logits_to_keep,
        )

        # Returned logits correspond only to the last logits_to_keep positions.
        shift_logits = outputs.logits[:, :-1, :]
        shift_labels = labels[:, -(logits_to_keep - 1):]
        valid_tokens = shift_labels.ne(-100)

        valid_logits = shift_logits[valid_tokens].float()
        valid_labels = shift_labels[valid_tokens]

        per_token_loss = F.cross_entropy(
            valid_logits,
            valid_labels,
            reduction="none",
        )

        batch_ids = (
            torch.arange(labels.size(0), device=labels.device)
            .unsqueeze(1)
            .expand_as(shift_labels)
        )
        valid_batch_ids = batch_ids[valid_tokens]

        weights = self.class_weights.to(labels.device)
        token_weights = weights[class_labels[valid_batch_ids]]

        loss = (per_token_loss * token_weights).sum() / token_weights.sum().clamp_min(1e-8)
        return (loss, outputs) if return_outputs else loss


In [ ]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.05,
    weight_decay=0.01,
    max_grad_norm=1.0,
    fp16=(compute_dtype == torch.float16),
    bf16=(compute_dtype == torch.bfloat16),
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=5,
    logging_first_step=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    remove_unused_columns=False,
    group_by_length=True,
    length_column_name="length",
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    report_to="tensorboard",
    logging_dir=str(LOG_DIR),
    run_name="gemma3_4b_accuracy_qlora",
    seed=42,
    data_seed=42,
)

trainer = TargetOnlyWeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=left_pad_collate,
    class_weights=class_weights.tolist(),
)

# Mandatory preflight: proves that the real DataLoader batch is >= 6 and bounded.
preflight_batch = next(iter(trainer.get_train_dataloader()))
print("Actual batch shape:", tuple(preflight_batch["input_ids"].shape))
print("Actual batch size:", preflight_batch["input_ids"].shape[0])
print("Sequence length:", preflight_batch["input_ids"].shape[1])
print("Supervised JSON tokens per row:", preflight_batch["labels"].ne(-100).sum(dim=1).tolist())

assert preflight_batch["input_ids"].shape[0] >= 6
assert preflight_batch["input_ids"].shape[1] <= MAX_LENGTH

torch.cuda.empty_cache()
gc.collect()


In [ ]:
train_result = trainer.train()

trainer.save_model(str(ADAPTER_DIR))
processor.save_pretrained(str(ADAPTER_DIR))

with open(METRICS_DIR / "trainer_log_history.json", "w") as handle:
    json.dump(trainer.state.log_history, handle, indent=2)

with open(METRICS_DIR / "run_config.json", "w") as handle:
    json.dump(
        {
            "model_id": MODEL_ID,
            "training_rows": len(df_train),
            "validation_rows": len(val_df),
            "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
            "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
            "effective_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
            "max_length": MAX_LENGTH,
            "epochs": NUM_TRAIN_EPOCHS,
            "learning_rate": LEARNING_RATE,
        },
        handle,
        indent=2,
    )

print("Adapter saved to:", ADAPTER_DIR)


In [ ]:
def predict(question, evaluation_answer, max_new_tokens=16):
    model.eval()
    model.config.use_cache = True

    prompt_ids = encode_prompt_ids(
        question,
        evaluation_answer,
        max_prompt_tokens=MAX_LENGTH,
    )
    input_ids = torch.tensor([prompt_ids], dtype=torch.long, device=model.device)
    attention_mask = torch.ones_like(input_ids)

    with torch.inference_mode():
        generated = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = generated[0, input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


torch.cuda.empty_cache()
predictions = []
raw_outputs = []

for row_number, (_, row) in enumerate(val_df.iterrows(), start=1):
    raw = predict(row[QUESTION_COL], row[ANSWER_COL])
    prediction = parse_prediction(raw)
    predictions.append(prediction)
    raw_outputs.append(raw)

    if row_number % 20 == 0 or row_number == len(val_df):
        print(f"Evaluated {row_number}/{len(val_df)}")

evaluation_df = val_df.copy()
evaluation_df["gemma_accuracy"] = predictions
evaluation_df["gemma_raw_output"] = raw_outputs
evaluation_df.to_excel(METRICS_DIR / "validation_predictions.xlsx", index=False)

valid = evaluation_df.dropna(subset=["gemma_accuracy"]).copy()
y_true = valid[LABEL_COL].astype(int)
y_pred = valid["gemma_accuracy"].astype(int)

final_accuracy = accuracy_score(y_true, y_pred)
final_macro_f1 = f1_score(y_true, y_pred, average="macro", labels=[0, 1, 2], zero_division=0)
parse_rate = len(valid) / len(evaluation_df)

print(f"Parse rate: {parse_rate:.3f}")
print(f"Accuracy: {final_accuracy:.3f}")
print(f"Macro-F1: {final_macro_f1:.3f}")
print(classification_report(y_true, y_pred, labels=[0, 1, 2], zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
print("Confusion matrix:")
print(cm)
np.savetxt(METRICS_DIR / "confusion_matrix.csv", cm, delimiter=",", fmt="%d")

errors = valid[y_true != y_pred].copy()
errors.to_excel(METRICS_DIR / "validation_errors.xlsx", index=False)

with open(METRICS_DIR / "final_metrics.json", "w") as handle:
    json.dump(
        {
            "parsed_examples": int(len(valid)),
            "total_examples": int(len(evaluation_df)),
            "parse_rate": float(parse_rate),
            "accuracy": float(final_accuracy),
            "macro_f1": float(final_macro_f1),
        },
        handle,
        indent=2,
    )


In [ ]:
logs = pd.DataFrame(trainer.state.log_history)
display(logs.tail(20))

if "loss" in logs.columns:
    train_logs = logs.dropna(subset=["loss"])
    plt.figure(figsize=(8, 5))
    plt.plot(train_logs["step"], train_logs["loss"], marker="o")
    plt.xlabel("Training step")
    plt.ylabel("Loss")
    plt.title("Gemma 3 4B QLoRA training loss")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(METRICS_DIR / "training_loss.png", dpi=200)
    plt.show()

%load_ext tensorboard
%tensorboard --logdir /content/active_learning_project/logs/gemma3_4b_accuracy


## Increasing the training sample size

`TRAIN_LIMIT = None` uses every labeled row in the training workbook. To run a controlled subset, set it to an integer such as `1500`. Increasing the number of rows does not increase peak GPU memory; batch size and sequence length control peak memory. It increases the number of optimizer steps and therefore training time.

If batch size 6 still fails on an unusually constrained runtime, do not silently reduce it. First confirm that the preflight prints a sequence length no greater than 256 and that the model footprint indicates 4-bit loading. Restart the runtime after every CUDA OOM before trying again.
